In [1]:
import numpy as np
import pandas as pd
from pylab import *
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import pandas as pd
from pylab import *
from sequana import FastA
import tensorflow as tf
from tensorflow.keras import layers, models
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import random
from collections import defaultdict
from sklearn.metrics import classification_report


2025-07-30 13:10:36.674652: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-30 13:10:36.679168: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-30 13:10:36.691138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753873836.711818 3878364 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753873836.717751 3878364 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753873836.733691 3878364 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
centromeres = pd.read_csv("../../output/estimation/donovani/donovani_genome.csv")
centromeres = {
    str(row['Chromosome']): (row['start'], row['end'])
    for _, row in centromeres.iterrows()
}


In [3]:
f = FastA("../../data/Fasta/genome.fasta")

In [4]:
def one_hot_encoding(x):
    if x == 'A':
        return np.array([1,0,0,0])
    elif x == 'C':
        return np.array([0,1,0,0])
    elif x == 'G':
        return np.array([0,0,1,0])
    elif x == 'T':
        return np.array([0,0,0,1])
    else:
        return np.array([0,0,0,0])
        

In [5]:
SIZE=3000

In [6]:
def get_true_positive(size=3000):
    data = []
    for chrom in range(1,36+1):
        start, stop = centromeres[str(chrom)]
        #seq = f.sequences[f.names.index(str(chrom))]
        if f.names[0] == "1":
            seq = f.sequences[f.names.index(str(chrom))]
        else:
            seq = f.sequences[chrom-1]
            
        if stop-start != 3000:
            stop = start + size
        # flip to get more positives
        data.append([one_hot_encoding(x) for x in seq[start:stop]])
        data.append([one_hot_encoding(x) for x in seq[start:stop][::-1]])
        

    return data
positives = get_true_positive(SIZE)

In [7]:
lengths = f.get_lengths_as_dict()


In [8]:

##################################### WARNING #############################################
############################### REMOVE FALSE NEGATIVE (centromeres) #######################

def get_true_negatives(N=1000,size=3000,seed=42):
    data = []
    positions = defaultdict(list)
    random.seed(seed)

    # get the random combos first
    for i in tqdm(range(N)):
        chrom = random.randint(1,36)
        N = lengths[str(chrom)]
        pos = random.randint(1, N-size)
        start, stop = centromeres[str(chrom)]
        if pos>start and pos<stop:
            pass # this is a centromeres so not a negative
        else:
            positions[chrom].append(pos)
        
        
    for chrom in tqdm(positions.keys()):
        seq = f.sequences[f.names.index(str(chrom))]
        for position in positions[chrom]:
            data.append([one_hot_encoding(x) for x in seq[position:position+size]])
    return data

    #negatives = get_true_negatives(N=10000, size=SIZE)

In [9]:
layers_values = [4,8,16,32,64]
for layer in layers_values:
    results = []
    f = FastA("../../data/Fasta/genome.fasta")    
    for i in range(1,11):
        print("###############################################################################")
        print(f'{i}/10')
        print("###############################################################################")

        negatives = get_true_negatives(N=10000, size=SIZE,seed=i)
     
        
        X_pos = positives
        X_neg = negatives
        
        
        # Create label arrays
        y_pos = [1] * len(X_pos)
        y_neg = [0] * len(X_neg)
        
        # Combine and shuffle
        X = np.array(X_pos + X_neg)  # shape: (N, 3000, 4)
        y = np.array(y_pos + y_neg)
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )
    
    
        layer_size=layer
        print(layer_size)
        print(SIZE)
        model = models.Sequential([
            layers.Input(shape=(SIZE, 4)),
            layers.Conv1D(layer_size, kernel_size=15, activation='relu'),
            layers.MaxPooling1D(pool_size=2),
            layers.Conv1D(layer_size*2, kernel_size=10, activation='relu'),
            layers.GlobalMaxPooling1D(),
            layers.Dense(layer_size, activation='relu'),
            layers.Dense(1, activation='sigmoid')  # binary classification
        ])
        model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        
        seed = 42
    
        tf.random.set_seed(seed)
        
        tf.config.experimental.enable_op_determinism()
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=80,
            batch_size=32,
            class_weight={0: 1, 1: len(y_neg)/len(y_pos)},  # handle imbalance
            callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
        )
    
    
        y_pred = model.predict(X_test) > 0.2
        report = classification_report(y_test, y_pred, output_dict=True)
    
        metrics = {
            'f1_score': report['1']['f1-score'],
            'precision': report['1']['precision'],
            'recall': report['1']['recall']
        }
        results.append(metrics)

    with open(f"{layer}_layer_result_patience_donovani.pkl", "wb") as f:
        pickle.dump(results, f)




###############################################################################
1/10
###############################################################################


100%|███████████████████████████████████████████| 36/36 [00:32<00:00,  1.12it/s]


4
3000


2025-07-29 16:55:34.594095: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/80


...

251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.7739 - loss: 1.5403

2025-07-29 23:46:30.520146: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


251/251 ━━━━━━━━━━━━━━━━━━━━ 73s 283ms/step - accuracy: 0.7725 - loss: 1.5401 - val_accuracy: 0.0070 - val_loss: 0.7415
Epoch 2/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 71s 281ms/step - accuracy: 0.3605 - loss: 1.2497 - val_accuracy: 0.9930 - val_loss: 0.2898
Epoch 3/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 71s 281ms/step - accuracy: 0.8130 - loss: 1.0403 - val_accuracy: 0.7859 - val_loss: 0.3658
Epoch 4/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 70s 281ms/step - accuracy: 0.8685 - loss: 0.6694 - val_accuracy: 0.9516 - val_loss: 0.1133
Epoch 5/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 82s 281ms/step - accuracy: 0.8983 - loss: 0.5036 - val_accuracy: 0.9870 - val_loss: 0.0334
Epoch 6/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 71s 281ms/step - accuracy: 0.9238 - loss: 0.3238 - val_accuracy: 0.9770 - val_loss: 0.0536
Epoch 7/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 82s 281ms/step - accuracy: 0.9476 - loss: 0.2816 - val_accuracy: 0.9870 - val_loss: 0.0339
Epoch 8/80
251/251 ━━━━━━━━━━━━━━━━━━━━ 71s 281ms/step - accuracy: 0.9822 - loss: 0.0823 - val

2025-07-30 00:13:13.401755: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 74ms/step


In [11]:
with open(f"{64}_layer_result_patience5.pkl", "wb") as f:
    pickle.dump(results, f)


In [32]:
import pickle
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Liste des tailles de couches
layer_sizes = [16, 32, 64, 128]

# Charger les résultats et les mettre dans une liste
all_data = []

for layer in layer_sizes: 
    with open(f"{layer}_layer_result.pkl", "rb") as f:
        results = pickle.load(f)
        for metrics in results:
            all_data.append({
                'layer_size': layer,
                'f1_score': metrics['f1_score'],
                'precision': metrics['precision'],
                'recall': metrics['recall']
            })

# Convertir en DataFrame
df = pd.DataFrame(all_data)

# Afficher les 3 boxplots
metrics = ['f1_score', 'precision', 'recall']
for metric in metrics:
    plt.figure(figsize=(8, 6))
    sns.boxplot(x='layer_size', y=metric, data=df)
    plt.title(f"Boxplot of {metric} by Layer Size")
    plt.xlabel("Layer Size")
    plt.ylabel(metric.capitalize())
    plt.grid(True)
    plt.tight_layout()
    plt.show()


(8011, 2003)